# Small SAGE-like CNN Weight VAE + Post-Hoc Latent Smoothing

Cache-first experiment for testing paper-like post-hoc decoder latent smoothing on a small VAE trained over TinyCNN weights. The post-hoc NF is trained only on frozen decoder geometry, never on downstream task loss or preconditioner targets.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.sage_cnn_vae_smoothing import (
    celo_meta_config,
    fast_config,
    paperish_config,
    run_or_load,
)

plt.rcParams['figure.dpi'] = 130


## Hardcoded Config

All experiment knobs are intentionally exposed in this notebook cell. Change `run_label` when changing a protocol so cache reuse is explicit. `FAST_CONFIG` is the default; switch `ACTIVE_CONFIG = PAPERISH_CONFIG` for a heavier TinyCNN run or `ACTIVE_CONFIG = CELO_META_CONFIG` for the Celo-style Adam/tau setup. Artifacts are written under `artifact_root/{run_label}`.


In [ ]:
# Cache / run identity
ARTIFACT_ROOT = 'artifacts/loss_landscape_analysis/sage_cnn_vae_smoothing'
DATA_ROOT = 'artifacts/loss_landscape_analysis/data'
DEVICE = 'cuda:0'
DTYPE = 'float32'
SEED = 0

FAST_CONFIG = fast_config(
    # run/cache
    run_label='sage_cnn_vae_smoothing_fast',
    artifact_root=ARTIFACT_ROOT,
    dataset_name='fashion_mnist',
    data_root=DATA_ROOT,
    download=True,
    cache_first=True,
    force_rerun=False,
    device=DEVICE,
    dtype=DTYPE,
    seed=SEED,
    show_progress=True,
    save_figures=True,

    # FashionMNIST subset used for TinyCNN training/eval
    train_subset=1024,
    test_subset=512,
    cnn_batch_size=64,

    # Stage 1: TinyCNN weight snapshot pool
    weight_runs=16,
    weight_train_steps=80,
    weight_snapshot_every=10,
    weight_lr=1e-3,
    vae_train_fraction=0.8,

    # Stage 2: small SAGE-like weight VAE
    latent_dim=16,
    vae_hidden_dim=512,
    vae_steps=500,
    vae_batch_size=64,
    vae_lr=1e-3,
    vae_loss_kind='big_vae',
    beta_kl=0.01,
    bigvae_operator_probe_rows=32,
    bigvae_patch_size=4,
    bigvae_behavioral_coef=1.0,
    bigvae_structural_coef=1.0,
    bigvae_bias_coef=1.0,
    bigvae_behavioral_lambda_operator=10.0,
    bigvae_behavioral_lambda_dir=1.0,
    bigvae_behavioral_lambda_scale=10.0,
    bigvae_behavioral_gamma=0.5,
    bigvae_behavioral_huber_delta=0.1,
    bigvae_struct_gamma=0.5,
    bigvae_struct_lambda_dir=1.0,
    bigvae_struct_lambda_scale=10.0,
    bigvae_struct_lambda_rec=0.0,
    bigvae_struct_lambda_rel=0.0,
    bigvae_struct_huber_delta=0.1,
    # If >0, the pipeline automatically trains two VAEs: baseline coeff=0 and regularized coeff=this value.
    vae_geometry_reg_coeff=0.0,
    vae_geometry_reg_samples=1,
    vae_geometry_reg_detach_latents=False,

    # Stage 3: post-hoc RQ-spline NF smoothing over frozen decoder geometry
    flow_steps=300,
    flow_batch_size=8,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_log_every=25,
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_spline_bins=8,
    flow_spline_bound=5.0,
    flow_eta=0.0,
    mixup_alpha_min=-0.1,
    mixup_alpha_max=1.1,
    geometry_eval_samples=16,

    # Stage 4: downstream optimization comparison
    tune_starts=4,
    eval_starts=8,
    downstream_steps=80,
    downstream_eval_every=1,
    downstream_batch_size=128,
    raw_lrs=(1e-4, 3e-4, 1e-3, 3e-3),
    latent_lrs=(1e-3, 3e-3, 1e-2, 3e-2),
    nf_lrs=(1e-3, 3e-3, 1e-2, 3e-2),
    success_threshold=0.25,
    finite_penalty=1e12,
)

PAPERISH_CONFIG = paperish_config(
    # run/cache
    run_label='sage_cnn_vae_smoothing_paperish',
    artifact_root=ARTIFACT_ROOT,
    dataset_name='fashion_mnist',
    data_root=DATA_ROOT,
    download=True,
    cache_first=True,
    force_rerun=False,
    device=DEVICE,
    dtype=DTYPE,
    seed=SEED,
    show_progress=True,
    save_figures=True,

    # FashionMNIST subset used for TinyCNN training/eval
    train_subset=4096,
    test_subset=1024,
    cnn_batch_size=64,

    # Stage 1: TinyCNN weight snapshot pool
    weight_runs=64,
    weight_train_steps=200,
    weight_snapshot_every=10,
    weight_lr=1e-3,
    vae_train_fraction=0.8,

    # Stage 2: small SAGE-like weight VAE
    latent_dim=16,
    vae_hidden_dim=512,
    vae_steps=2000,
    vae_batch_size=64,
    vae_lr=1e-3,
    vae_loss_kind='big_vae',
    beta_kl=0.01,
    bigvae_operator_probe_rows=32,
    bigvae_patch_size=4,
    bigvae_behavioral_coef=1.0,
    bigvae_structural_coef=1.0,
    bigvae_bias_coef=1.0,
    bigvae_behavioral_lambda_operator=10.0,
    bigvae_behavioral_lambda_dir=1.0,
    bigvae_behavioral_lambda_scale=10.0,
    bigvae_behavioral_gamma=0.5,
    bigvae_behavioral_huber_delta=0.1,
    bigvae_struct_gamma=0.5,
    bigvae_struct_lambda_dir=1.0,
    bigvae_struct_lambda_scale=10.0,
    bigvae_struct_lambda_rec=0.0,
    bigvae_struct_lambda_rel=0.0,
    bigvae_struct_huber_delta=0.1,
    # If >0, the pipeline automatically trains two VAEs: baseline coeff=0 and regularized coeff=this value.
    vae_geometry_reg_coeff=0.0,
    vae_geometry_reg_samples=1,
    vae_geometry_reg_detach_latents=False,

    # Stage 3: post-hoc RQ-spline NF smoothing over frozen decoder geometry
    flow_steps=1000,
    flow_batch_size=8,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_log_every=25,
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_spline_bins=8,
    flow_spline_bound=5.0,
    flow_eta=0.2,
    mixup_alpha_min=-0.1,
    mixup_alpha_max=1.1,
    geometry_eval_samples=64,

    # Stage 4: downstream optimization comparison
    tune_starts=8,
    eval_starts=16,
    downstream_steps=200,
    downstream_eval_every=1,
    downstream_batch_size=128,
    raw_lrs=(1e-4, 3e-4, 1e-3, 3e-3),
    latent_lrs=(1e-3, 3e-3, 1e-2, 3e-2),
    nf_lrs=(1e-3, 3e-3, 1e-2, 3e-2),
    success_threshold=0.25,
    finite_penalty=1e12,
)

CELO_META_CONFIG = celo_meta_config(
    run_label='sage_cnn_vae_smoothing_celo_meta_tinybigvae_v2',
    artifact_root=ARTIFACT_ROOT,
    data_root=DATA_ROOT,
    download=True,
    cache_first=True,
    force_rerun=True,
    device=DEVICE,
    dtype=DTYPE,
    seed=SEED,
    show_progress=True,
    save_figures=True,
    celo_tasks=('mnist', 'fashion_mnist', 'svhn', 'cifar10'),
    celo_image_size=8,
    celo_hidden_dim=32,
    celo_tau_min=1e-3,
    celo_tau_max=1e3,
    celo_adam_lrs=(1e-4, 3e-4, 1e-3, 3e-3, 1e-2),
    latent_dim=8,
    vae_hidden_dim=32,
    vae_arch='tiny_big_vae',
    tiny_bigvae_patch_size=16,
    tiny_bigvae_token_dim=16,
    tiny_bigvae_pos_dim=8,
    tiny_bigvae_resampler_latents=2,
    tiny_bigvae_attention_heads=1,
)

ACTIVE_CONFIG = CELO_META_CONFIG
ACTIVE_CONFIG


## Run Or Load


In [ ]:
tables = run_or_load(ACTIVE_CONFIG)
output_dir = Path(tables.output_dir)
print('output_dir:', output_dir)
print('vae_metrics:', tables.vae_metrics.shape)
print('geometry:', tables.geometry.shape)
print('selected_lrs:', tables.selected_lrs.shape)
print('downstream_results:', tables.downstream_results.shape)
print('downstream_curves:', tables.downstream_curves.shape)


## VAE Reconstruction Quality


In [ ]:
vae_quality = tables.vae_metrics[tables.vae_metrics.get('record_type', '') == 'vae_quality'].copy()
vae_history = tables.vae_metrics[tables.vae_metrics.get('record_type', '') == 'vae_train_history'].copy()

if not vae_quality.empty:
    display(vae_quality[[
        'reconstruction_mse',
        'reconstruction_rel_l2',
        'latent_roundtrip_l2',
        'raw_test_acc',
        'decoded_test_acc',
        'raw_test_loss',
        'decoded_test_loss',
    ]].describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if not vae_history.empty:
    variant_groups = vae_history.groupby('vae_variant') if 'vae_variant' in vae_history.columns else [('baseline', vae_history)]
    for variant, rows in variant_groups:
        axes[0].plot(rows['step'], rows['train_recon_mse'], label=f'{variant} train recon')
        axes[0].plot(rows['step'], rows['val_recon_mse'], linestyle='--', label=f'{variant} val recon')
    axes[0].set_yscale('log')
    axes[0].legend()
if not vae_quality.empty:
    variant_groups = vae_quality.groupby('vae_variant') if 'vae_variant' in vae_quality.columns else [('baseline', vae_quality)]
    for variant, rows in variant_groups:
        axes[1].scatter(rows['raw_test_acc'], rows['decoded_test_acc'], label=str(variant), alpha=0.75)
    lo = min(float(vae_quality['raw_test_acc'].min()), float(vae_quality['decoded_test_acc'].min()))
    hi = max(float(vae_quality['raw_test_acc'].max()), float(vae_quality['decoded_test_acc'].max()))
    axes[1].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[1].legend()
axes[0].set_title('VAE normalized reconstruction MSE')
axes[0].set_xlabel('step')
axes[0].grid(True, alpha=0.25)
axes[1].set_title('decoded vs raw test accuracy')
axes[1].set_xlabel('raw')
axes[1].set_ylabel('decoded')
axes[1].grid(True, alpha=0.25)
path = output_dir / 'vae_reconstruction_quality.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Post-Hoc Geometry


In [ ]:
display(tables.geometry)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
geom = tables.geometry.copy()
geom['plot_label'] = geom['coordinate']
if 'vae_variant' in geom.columns:
    geom['plot_label'] = geom['vae_variant'].astype(str) + '/' + geom['coordinate'].astype(str)
geom = geom.set_index('plot_label')
geom['isometry_objective'].plot(kind='bar', ax=axes[0])
geom['condition_median'].plot(kind='bar', ax=axes[1])
geom['log_eig_spread_median'].plot(kind='bar', ax=axes[2])
axes[0].set_title('relaxed isometry objective')
axes[1].set_title('median pullback condition')
axes[2].set_title('median log eig spread')
for ax in axes:
    ax.grid(True, axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=35)
path = output_dir / 'posthoc_geometry.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Learning Rates And Downstream Metrics


In [ ]:
selected = tables.selected_lrs.copy()
sort_cols = ['method', 'selected']
sort_ascending = [True, False]
if 'vae_variant' in selected.columns:
    sort_cols = ['vae_variant'] + sort_cols
    sort_ascending = [True] + sort_ascending
display(selected.sort_values(sort_cols, ascending=sort_ascending))

results = tables.downstream_results.copy()
group_cols = ['method']
if 'vae_variant' in results.columns:
    group_cols = ['vae_variant', 'method']
summary = results.groupby(group_cols, as_index=False).agg(
    median_aulc=('aulc', 'median'),
    median_final_train_loss=('final_train_loss', 'median'),
    median_best_train_loss=('best_train_loss', 'median'),
    median_final_test_loss=('final_test_loss', 'median'),
    median_final_test_acc=('final_test_acc', 'median'),
    median_reconstruction_rel_l2=('reconstruction_rel_l2', 'median'),
    diverged_rate=('diverged', 'mean'),
)
display(summary.sort_values('median_aulc'))


## Downstream Curves


In [ ]:
curves = tables.downstream_curves.copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
group_cols = ['method']
if 'vae_variant' in curves.columns:
    group_cols = ['vae_variant', 'method']
for key, rows in curves.groupby(group_cols):
    label = '/'.join(str(part) for part in key) if isinstance(key, tuple) else str(key)
    train_pivot = rows.pivot(index='step', columns='start_index', values='train_loss')
    acc_pivot = rows.pivot(index='step', columns='start_index', values='test_acc')
    axes[0].plot(train_pivot.index, train_pivot.median(axis=1), label=label)
    axes[1].plot(acc_pivot.index, acc_pivot.median(axis=1), label=label)
axes[0].set_yscale('log')
axes[0].set_title('median train loss')
axes[1].set_title('median test accuracy')
for ax in axes:
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
path = output_dir / 'downstream_curves.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Paired AULC And Random-NF Control


In [ ]:
pivot_index = ['start_index']
if 'vae_variant' in results.columns:
    pivot_index = ['vae_variant', 'start_index']
pivot = results.pivot_table(index=pivot_index, columns='method', values='aulc', aggfunc='median')
display(pivot)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
if {'decoder_latent', 'decoder_trained_nf'}.issubset(pivot.columns):
    axes[0].scatter(pivot['decoder_latent'], pivot['decoder_trained_nf'])
    lo = float(min(pivot['decoder_latent'].min(), pivot['decoder_trained_nf'].min()))
    hi = float(max(pivot['decoder_latent'].max(), pivot['decoder_trained_nf'].max()))
    axes[0].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[0].set_xlabel('decoder latent AULC')
    axes[0].set_ylabel('trained NF AULC')
if {'decoder_random_nf', 'decoder_trained_nf'}.issubset(pivot.columns):
    axes[1].scatter(pivot['decoder_random_nf'], pivot['decoder_trained_nf'])
    lo = float(min(pivot['decoder_random_nf'].min(), pivot['decoder_trained_nf'].min()))
    hi = float(max(pivot['decoder_random_nf'].max(), pivot['decoder_trained_nf'].max()))
    axes[1].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[1].set_xlabel('random NF AULC')
    axes[1].set_ylabel('trained NF AULC')
for ax in axes:
    ax.set_title('lower is better')
    ax.grid(True, alpha=0.25)
path = output_dir / 'paired_aulc.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Interpretation


In [ ]:
display(Markdown(tables.interpretation_markdown))
print((output_dir / 'interpretation.md').read_text(encoding='utf-8'))


## Files Written


In [ ]:
for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(path)
